# 02 — Training Run

Reproducible training of the best configuration (Experiment 9). Seeds are pinned.

In [ ]:
import sys, os
sys.path.append('..')

import random
import numpy as np
import torch
import yaml

# Pin seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded:', cfg)

In [ ]:
from src.data import load_dataloaders
train_dl, val_dl, test_dl = load_dataloaders(cfg)

In [ ]:
from src.model import SentimentClassifier

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentimentClassifier(
    checkpoint=cfg['model']['checkpoint'],
    dropout=cfg['model']['dropout'],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
import torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['training']['learning_rate'])
loss_fn = nn.CrossEntropyLoss()
epochs = cfg['training']['epochs']

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            preds = model(ids, mask).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch in train_dl:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        logits = model(ids, mask)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        losses.append(loss.item())
    val_acc = evaluate(model, val_dl)
    val_accs.append(val_acc)
    print(f'Epoch {epoch+1}/{epochs} | loss: {epoch_loss/len(train_dl):.4f} | val acc: {val_acc:.4f}')

In [ ]:
test_acc = evaluate(model, test_dl)
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
assert test_acc > 0.92, 'Expected >= 92%'

os.makedirs('../checkpoints', exist_ok=True)
torch.save(model.state_dict(), '../checkpoints/best.pt')
print('Checkpoint saved.')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(losses)
ax1.set_title('Training Loss'); ax1.set_xlabel('step'); ax1.set_ylabel('CE loss')
ax2.plot(val_accs, marker='o')
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy')
plt.tight_layout()
plt.savefig('../results/figures/training_curves.png', dpi=120)
plt.show()